# NB01: Data Collection and Cleaning

## Welfare Scheme Participation Analysis â€” Real Data

This notebook loads and merges three real, publicly available datasets:
1. **Census 2011** (640 districts, 118 features) â€” demographic baseline
2. **NFHS-5 2019-21** (706 districts, 109 features) â€” health & welfare indicators
3. **MGNREGA 2019-20** (681 districts, 46 features) â€” rural employment scheme participation

No synthetic data is used. All datasets are from public sources (Kaggle mirrors of data.gov.in, IIPS, and NDAP).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.data_loader import load_all_data
from src.data_cleaner import (
    build_master_dataset, clean_census, clean_nfhs, clean_mgnrega,
    derive_census_features, derive_nfhs_features, normalize_state_name, normalize_district_name
)
from src.visualization import setup_plot_style, save_figure

setup_plot_style()
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. Load Raw Data

In [ ]:
census, nfhs, mgnrega, mgnrega_all, report = load_all_data(mgnrega_year='2019-20')
print(report.to_string(index=False))

## 2. Inspect Raw Datasets

In [ ]:
print('=== Census 2011 ===')
print(f'Shape: {census.shape}')
print(f'States: {census["state"].nunique()}, Districts: {census["district"].nunique()}')
print(f'Missing values: {census.isna().sum().sum()}')
census.head(3)

In [ ]:
print('=== NFHS-5 ===')
print(f'Shape: {nfhs.shape}')
print(f'States: {nfhs["state"].nunique()}, Districts: {nfhs["district"].nunique()}')
print(f'Missing values: {nfhs.isna().sum().sum()}')
nfhs.head(3)

In [ ]:
print('=== MGNREGA 2019-20 ===')
print(f'Shape: {mgnrega.shape}')
print(f'States: {mgnrega["state"].nunique()}, Districts: {mgnrega["district"].nunique()}')
print(f'Key columns: hh_demanded_work, hh_worked, persondays_total, labour_expenditure')
mgnrega.head(3)

## 3. Name Normalization & Overlap Analysis

In [ ]:
census_clean = derive_census_features(clean_census(census))
nfhs_clean = derive_nfhs_features(clean_nfhs(nfhs))
mgnrega_clean = clean_mgnrega(mgnrega)

census_keys = set(zip(census_clean['state'], census_clean['district']))
nfhs_keys = set(zip(nfhs_clean['state'], nfhs_clean['district']))
mgnrega_keys = set(zip(mgnrega_clean['state'], mgnrega_clean['district']))

all3 = census_keys & nfhs_keys & mgnrega_keys
c_n = census_keys & nfhs_keys
print(f'Districts in Census âˆ© NFHS-5: {len(c_n)}')
print(f'Districts in Census âˆ© NFHS-5 âˆ© MGNREGA: {len(all3)}')
print(f'Census only: {len(census_keys - nfhs_keys - mgnrega_keys)}')
print(f'NFHS-5 only: {len(nfhs_keys - census_keys - mgnrega_keys)}')
print(f'MGNREGA only: {len(mgnrega_keys - census_keys - nfhs_keys)}')

## 4. Build Master Dataset

In [ ]:
master = build_master_dataset(census, nfhs, mgnrega)
print(f'Master dataset: {master.shape[0]} districts Ã— {master.shape[1]} features')

In [ ]:
# Key feature columns
key_features = [
    'state', 'district', 'population_total', 'literacy_rate', 'rural_pct',
    'sc_pct', 'st_pct', 'sex_ratio', 'below_poverty_pct',
    'women_literacy_pct', 'health_insurance_pct', 'institutional_births_pct',
    'hh_demanded_work', 'hh_worked', 'persondays_total', 'labour_expenditure',
]
existing = [c for c in key_features if c in master.columns]
print(f'Key features present: {len(existing)}/{len(key_features)}')
master[existing].describe()

## 5. Missing Data Assessment

In [ ]:
missing = master.isna().sum()
missing_pct = (missing / len(master) * 100).round(1)
missing_report = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_report = missing_report[missing_report['missing'] > 0].sort_values('missing', ascending=False)
print(f'Columns with missing values: {len(missing_report)}')
if len(missing_report) > 0:
    print(missing_report.head(20))

## 6. Data Quality Summary

In [ ]:
print('='*60)
print('DATA QUALITY SUMMARY')
print('='*60)
print(f'Total districts in master: {len(master)}')
print(f'Total features: {master.shape[1]}')
print(f'States/UTs covered: {master["state"].nunique()}')
print(f'Districts with MGNREGA data: {(master["hh_demanded_work"] > 0).sum()}')
print(f'Districts with health insurance data: {master["health_insurance_pct"].notna().sum()}')
print(f'Districts with Census demographics: {master["population_total"].notna().sum()}')
print(f'All data from real public sources â€” zero synthetic values')
print('='*60)